# SQL Aggregations and GROUP BY — Examples

This notebook uses Python's built-in `sqlite3` module and pandas to display results. Run cells from top to bottom.

In [1]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
cursor = connection.cursor()
print("SQLite connection ready")

SQLite connection ready


## 1. Create and populate sample tables

In [2]:
cursor.executescript('''
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    department TEXT NOT NULL,
    role TEXT NOT NULL,
    salary REAL,
    location TEXT
);
INSERT INTO employees VALUES
(1, 'Asha', 'Engineering', 'Developer', 72000, 'Indore'),
(2, 'Ravi', 'Engineering', 'Developer', 68000, 'Pune'),
(3, 'Meera', 'Engineering', 'QA Analyst', 61000, 'Indore'),
(4, 'Kabir', 'Engineering', 'DevOps Engineer', 85000, 'Pune'),
(5, 'Neha', 'Sales', 'Sales Executive', 52000, 'Mumbai'),
(6, 'Arjun', 'Sales', 'Sales Executive', 48000, 'Indore'),
(7, 'Isha', 'Sales', 'Sales Manager', 78000, 'Mumbai'),
(8, 'Dev', 'HR', 'HR Executive', 42000, 'Pune'),
(9, 'Tara', 'HR', 'HR Manager', 65000, 'Indore'),
(10, 'Omar', 'Finance', 'Accountant', 58000, 'Mumbai'),
(11, 'Zoya', 'Finance', 'Analyst', 63000, 'Pune'),
(12, 'Nikhil', 'Finance', 'Analyst', NULL, 'Indore');

CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    region TEXT NOT NULL,
    customer_id INTEGER
);
INSERT INTO sales VALUES
(1, 'Laptop', 2, 650, 'West', 101),
(2, 'Mouse', 10, 20, 'West', 102),
(3, 'Laptop', 1, 650, 'North', 103),
(4, 'Keyboard', 4, 45, 'South', 101),
(5, 'Mouse', 5, 20, 'North', 104),
(6, 'Monitor', 3, 180, 'West', 105),
(7, 'Keyboard', 2, 45, 'West', 106),
(8, 'Monitor', 1, 180, 'South', 102),
(9, 'Laptop', 1, 650, 'South', 107),
(10, 'Mouse', 8, 20, 'South', NULL);
''')
print("Tables created and sample rows inserted")

Tables created and sample rows inserted


## 2. Inspect the data

In [3]:
pd.read_sql_query("SELECT * FROM employees ORDER BY employee_id", connection)

,employee_id,employee_name,department,role,salary,location
0,1,Asha,Engineering,Developer,72000.0,Indore
1,2,Ravi,Engineering,Developer,68000.0,Pune
2,3,Meera,Engineering,QA Analyst,61000.0,Indore
3,4,Kabir,Engineering,DevOps Engineer,85000.0,Pune
4,5,Neha,Sales,Sales Executive,52000.0,Mumbai
5,6,Arjun,Sales,Sales Executive,48000.0,Indore
6,7,Isha,Sales,Sales Manager,78000.0,Mumbai
7,8,Dev,HR,HR Executive,42000.0,Pune
8,9,Tara,HR,HR Manager,65000.0,Indore
9,10,Omar,Finance,Accountant,58000.0,Mumbai


In [4]:
pd.read_sql_query("SELECT * FROM sales ORDER BY sale_id", connection)

,sale_id,product,quantity,unit_price,region,customer_id
0,1,Laptop,2,650.0,West,101.0
1,2,Mouse,10,20.0,West,102.0
2,3,Laptop,1,650.0,North,103.0
3,4,Keyboard,4,45.0,South,101.0
4,5,Mouse,5,20.0,North,104.0
5,6,Monitor,3,180.0,West,105.0
6,7,Keyboard,2,45.0,West,106.0
7,8,Monitor,1,180.0,South,102.0
8,9,Laptop,1,650.0,South,107.0
9,10,Mouse,8,20.0,South,NaN


## 3. Aggregate functions

In [5]:
query = """SELECT COUNT(*) AS employee_count, COUNT(salary) AS employees_with_salary, SUM(salary) AS total_salary, ROUND(AVG(salary), 2) AS average_salary, MIN(salary) AS lowest_salary, MAX(salary) AS highest_salary FROM employees;"""
pd.read_sql_query(query, connection)

,employee_count,employees_with_salary,total_salary,average_salary,lowest_salary,highest_salary
0,12,11,692000.0,62909.09,42000.0,85000.0


## 4. GROUP BY one column

In [6]:
pd.read_sql_query("""SELECT department, COUNT(*) AS employee_count, ROUND(AVG(salary), 2) AS average_salary FROM employees GROUP BY department ORDER BY department;""", connection)

,department,employee_count,average_salary
0,Engineering,4,71500.00
1,Finance,3,60500.00
2,HR,2,53500.00
3,Sales,3,59333.33


## 5. GROUP BY multiple columns

In [7]:
pd.read_sql_query("""SELECT department, role, COUNT(*) AS employee_count, ROUND(AVG(salary), 2) AS average_salary FROM employees GROUP BY department, role ORDER BY department, role;""", connection)

,department,role,employee_count,average_salary
0,Engineering,DevOps Engineer,1,85000.0
1,Engineering,Developer,2,70000.0
2,Engineering,QA Analyst,1,61000.0
3,Finance,Accountant,1,58000.0
4,Finance,Analyst,2,63000.0
5,HR,HR Executive,1,42000.0
6,HR,HR Manager,1,65000.0
7,Sales,Sales Executive,2,50000.0
8,Sales,Sales Manager,1,78000.0


## 6. WHERE filters rows before grouping

In [8]:
pd.read_sql_query("""SELECT department, COUNT(*) AS qualifying_employees, ROUND(AVG(salary), 2) AS average_salary FROM employees WHERE salary >= 60000 GROUP BY department ORDER BY average_salary DESC;""", connection)

,department,qualifying_employees,average_salary
0,Sales,1,78000.0
1,Engineering,4,71500.0
2,HR,1,65000.0
3,Finance,1,63000.0


## 7. HAVING filters groups after aggregation

In [9]:
pd.read_sql_query("""SELECT department, COUNT(*) AS employee_count, ROUND(AVG(salary), 2) AS average_salary FROM employees GROUP BY department HAVING COUNT(*) >= 2 ORDER BY average_salary DESC;""", connection)

,department,employee_count,average_salary
0,Engineering,4,71500.00
1,Finance,3,60500.00
2,Sales,3,59333.33
3,HR,2,53500.00


## 8. Conditional aggregation

In [10]:
pd.read_sql_query("""SELECT department, COUNT(*) AS total_employees, SUM(CASE WHEN salary >= 60000 THEN 1 ELSE 0 END) AS employees_earning_60000_plus FROM employees GROUP BY department ORDER BY department;""", connection)

,department,total_employees,employees_earning_60000_plus
0,Engineering,4,4
1,Finance,3,1
2,HR,2,1
3,Sales,3,1


## 9. Aggregate sales revenue

In [11]:
pd.read_sql_query("""SELECT product, SUM(quantity) AS units_sold, SUM(quantity * unit_price) AS revenue, ROUND(AVG(unit_price), 2) AS average_unit_price FROM sales GROUP BY product ORDER BY revenue DESC, product ASC;""", connection)

,product,units_sold,revenue,average_unit_price
0,Laptop,4,2600.0,650.0
1,Monitor,4,720.0,180.0
2,Mouse,23,460.0,20.0
3,Keyboard,6,270.0,45.0


## 10. GROUP BY region and filter with HAVING

In [12]:
pd.read_sql_query("""SELECT region, SUM(quantity * unit_price) AS revenue FROM sales GROUP BY region HAVING SUM(quantity * unit_price) > 1000 ORDER BY revenue DESC;""", connection)

,region,revenue
0,West,2130.0
1,South,1170.0


## 11. DISTINCT counts and NULL behavior

In [13]:
pd.read_sql_query("""SELECT COUNT(*) AS sales_rows, COUNT(customer_id) AS rows_with_customer, COUNT(DISTINCT customer_id) AS unique_customers FROM sales;""", connection)

,sales_rows,rows_with_customer,unique_customers
0,10,9,7


## 12. Try a different threshold

In [14]:
pd.read_sql_query("""SELECT department, SUM(salary) AS total_payroll FROM employees GROUP BY department HAVING SUM(salary) > 100000 ORDER BY total_payroll DESC;""", connection)

,department,total_payroll
0,Engineering,286000.0
1,Sales,178000.0
2,Finance,121000.0
3,HR,107000.0


## Key takeaways

- `GROUP BY` defines the level of detail of each output row.
- `WHERE` filters source rows; `HAVING` filters groups.
- `COUNT(*)` counts rows, while `COUNT(column)` ignores `NULL` values.
- Use `CASE` within aggregates for conditional counts and totals.
- Confirm that the metric matches the business question.